In [ ]:
# -*- coding: utf-8 -*-

from skimage.transform import rescale, resize
from skimage.io import imshow,imread,show,imsave
import numpy as np
from pathlib import Path
import scipy.io
import os 
import nibabel as nib


def FindPos (proj,dim):
    if len(np.shape(proj))>=2:
        col_zeros=np.where(proj==0)[1]
        selected_col=np.bincount(col_zeros)==dim
        pos=np.where(selected_col==False)[0]
        pos_dim=[]
        pos_dim.append(pos[0]);
        pos_dim.append(pos[-1]);
        
    return pos_dim

def findCrop (image):
    #image=np.transpose(image,(2,0,1))#The image must be in (T,H,W) format. Delete this line if it is already like that.
    size=np.shape(image)
    dim=[0,0] if len(size)<3 else [0,2,0]
    pos=[]
    for i in range(0,len(dim)):
        proj=np.transpose(np.sum(image,i)) if i==1 else np.sum(image,i)
        pos.append(FindPos(proj,size[dim[i]]))
    print(pos)

    return pos

def crop (image,pos): 
    #image=np.transpose(image,(2,0,1))#The image must be in (T,H,W) format. Delete this line if it is already like that.
    final_image=image[pos[1][0]:pos[1][1],pos[2][0]:pos[2][1],pos[0][0]:pos[0][1]] if len(np.shape(image))>=3 else image[pos[0][1]:pos[0][1],pos[1][0]:pos[1][0]]
    ##final_image=np.transpose(final_image,(1,2,0))#To return to (H,W,T) format.
    return final_image





In [ ]:
#CROP FUNCTION
input_folder=r'DATA_ROOT\3_StructuralPreProcessing\preprocesado_results_v2'
output_folder=r'DATA_ROOT\3_StructuralPreProcessing\preprocesado_results_v2'
mod_list=['T1W_CE_3D_FFE_TRA_IQ','T1W_3D_FFE_TRA_IQ', 'FLAIR_3D_SPIR_TRA_IQ', 'SWIm']
template='T1W_CE_3D_FFE_TRA_IQ'
for pat in range(98,122):
    for mod in mod_list:
        print(mod)
        if mod==template:
            pat_number='PAT_'+str(pat)+'_'
            print(pat_number)
            path=os.path.join(input_folder,mod,pat_number, pat_number+mod+'_brainExtraction.nii.gz')
            im_load=nib.load(path)
            im=im_load.get_fdata()
            pos_vect=findCrop(im)
            im_crop=crop(im,pos_vect)
            print('MATRIX SIZE BEFORE CROPPING:'+str(im.shape))
            print('MATRIX SIZE AFTER CROPPING:'+str(im_crop.shape))
            print(pos_vect)
            imshow(np.sum(im_crop,0),cmap='gray')
            show()
            imshow(np.sum(im_crop,1),cmap='gray')
            show()
            imshow(np.sum(im_crop,2),cmap='gray')
            show()
            nifti_img=nib.Nifti1Image(im_crop, im_load.affine, im_load.header)
            nib.save(nifti_img,os.path.join(output_folder,mod,pat_number,pat_number+mod+'_final_crop.nii.gz'))
        else:
            path=os.path.join(input_folder,mod,pat_number, pat_number+mod+'_registered.nii.gz')
            im_load=nib.load(path)
            im=im_load.get_fdata()
            crop_img=crop(im,pos_vect)
            print('MATRIX SIZE BEFORE CROPPING:'+str(im.shape))
            print('MATRIX SIZE AFTER CROPPING:'+str(crop_img.shape))
            print(pos_vect)
            imshow(np.sum(im_crop,0),cmap='gray')
            show()
            imshow(np.sum(im_crop,1),cmap='gray')
            show()
            imshow(np.sum(im_crop,2),cmap='gray')
            show()
            #imshow(crop_img[100,:,:])
            nifti_img=nib.Nifti1Image(crop_img, im_load.affine, im_load.header)
            nib.save(nifti_img,os.path.join(output_folder,mod,pat_number,pat_number+mod+'_final_crop.nii.gz'))


